# LCA Lasers4MaaS: joining processes

Unit processes of the joining step for 5 use cases, with State of the Art (SoA) technologies and with Dynamic Beam Shaping (DBS).
The functional unit is 1 joined unit (`1 unit`).

The calculation code is in the `l4m_lca` package. This notebook holds the parameters and shows the results.

In [ ]:
import bw2data as bd
from IPython.display import display

from l4m_lca import plots
from l4m_lca.constants import EU27, EUROSTAT_ELEC_PRICE_2025S1
from l4m_lca.impacts import BackgroundImpacts
from l4m_lca.joining import joining_processes, technology_of
from l4m_lca.lcia import COST_COLUMN, ELECTRICITY_COLUMN, export, scenario_table
from l4m_lca.project import DEFAULT_INDICATORS, category, ef31_methods, find_activity, method_units, open_project
from l4m_lca.regional import electricity_markets, regional_table

## Parameters

In [ ]:
PROJECT_NAME = "Lasers4MaaS"
ECOINVENT_DATABASE_NAME = "ecoinvent-3.12-cutoff"
RESULTS_DIR = "results"

SOA_NAME = "State_of_the_Art"
DBS_NAME = "Dynamic_Beam_Shaping"

INDICATORS = DEFAULT_INDICATORS            # EF 3.1 indicators (substrings of the method names)

# Background activities: (name, location)
ELECTRICITY = ("market group for electricity, medium voltage", "RER")
SHIELDING_GAS = ("market for argon, liquid", "RER")

# Prices of the main resource inputs
COST_FACTORS = {
    "electricity_EUR_per_kWh": 0.29,   # industrial electricity
    "argon_EUR_per_kg": 5.87,          # argon gas
}

# Regional analysis: industrial electricity price of each country (EUR/kWh); argon keeps its price above
ELEC_PRICE = EUROSTAT_ELEC_PRICE_2025S1
REGIONAL_BASELINE = "RER"

In [ ]:
# Joining parameters of the SoA processes:
# joining power and welding station power (kW), joining speed (m/s), joint length (m), gas flow (kg/s).
# minimum = lowest power and gas flow at the fastest speed; maximum = highest at the slowest speed.
SOA_JOINING_PARAMS = {
    "Heat exchanger": {
        "power_kW_joining": {"min": 1.4, "max": 1.5},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.012, "max": 0.014},
        "joint_length_m": 42.9,
        "gas_flow_kg_per_s": {"min": 0.002676, "max": 0.002676},
    },
    "Fuel tank": {
        "power_kW_joining": {"min": 1.8, "max": 2.52},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.0025, "max": 0.0025},
        "joint_length_m": 5.06,
        "gas_flow_kg_per_s": {"min": 0.000268, "max": 0.000268},
    },
    "Heterogeneous pipe": {
        "power_kW_joining": {"min": 1.4, "max": 2.1},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.015, "max": 0.025},
        "joint_length_m": 0.094,
        "gas_flow_kg_per_s": {"min": 0.000892, "max": 0.000892},
    },
    "Freeze Dryers Shelf": {
        "power_kW_joining": {"min": 1.8, "max": 2.0},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.009, "max": 0.009},
        "joint_length_m": 13.2,
        "gas_flow_kg_per_s": {"min": 0.001487, "max": 0.001487},
    },
    "Battery cooling plate": {
        "power_kW_joining": {"min": 0, "max": 0},
        "power_kW_station": {"min": 2000, "max": 2000},
        "joining_speed_m_per_s": {"min": 0.460, "max": 0.460},
        "joint_length_m": 27.5,
        "gas_flow_kg_per_s": {"min": 0.123889, "max": 0.123889},
    },
}

# SoA technology of each use case
SOA_TECHNOLOGY = {
    "Heat exchanger": "standard laser",
    "Fuel tank": "TIG",
    "Heterogeneous pipe": "standard laser",
    "Freeze Dryers Shelf": "standard laser",
    "Battery cooling plate": "controlled atmosphere brazing",
}

In [ ]:
# Joining parameters with DBS. power_kW_station includes the chiller.
DBS_JOINING_PARAMS = {
    "Heat exchanger": {
        "power_kW_joining": {"min": 1.26, "max": 1.35},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 42.9,
        "gas_flow_kg_per_s": {"min": 0.002676, "max": 0.002676},
    },
    "Fuel tank": {
        "power_kW_joining": {"min": 1.62, "max": 2.268},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 5.06,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
    },
    "Heterogeneous pipe": {
        "power_kW_joining": {"min": 1.26, "max": 1.89},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.025, "max": 0.025},
        "joint_length_m": 0.094,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
    },
    "Freeze Dryers Shelf": {
        "power_kW_joining": {"min": 1.62, "max": 1.8},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joining_speed_m_per_s": {"min": 0.1, "max": 0.1},
        "joint_length_m": 13.2,
        "gas_flow_kg_per_s": {"min": 0.000595, "max": 0.000595},
    },
    "Battery cooling plate": {
        "power_kW_joining": {"min": 8, "max": 10},
        "power_kW_station": {"min": 21.7, "max": 21.7},
        "joint_length_m": 27.5,
        "joining_speed_m_per_s": {"min": 0.4, "max": 0.4},
        "gas_flow_kg_per_s": {"min": 0, "max": 0},
    },
}

## Project and background data

In [ ]:
ei = open_project(PROJECT_NAME, ECOINVENT_DATABASE_NAME)
elec = find_activity(ECOINVENT_DATABASE_NAME, *ELECTRICITY)
gas = find_activity(ECOINVENT_DATABASE_NAME, *SHIELDING_GAS)

methods = ef31_methods(INDICATORS)
units = method_units(methods)
for m in methods:
    print(f"  {m}  ->  unit: {units[category(m)]}")

impacts = BackgroundImpacts(methods, elec)
prices = {elec.key: COST_FACTORS["electricity_EUR_per_kWh"], gas.key: COST_FACTORS["argon_EUR_per_kg"]}

## Unit processes

Electricity and argon exchanges come from the parameters above. Each exchange has an amount (the midpoint) and a minimum and maximum for the scenarios.

In [ ]:
soa = bd.Database(SOA_NAME)
soa.write(joining_processes(SOA_NAME, SOA_JOINING_PARAMS, SOA_TECHNOLOGY, elec, gas))

dbs = bd.Database(DBS_NAME)
dbs.write(joining_processes(DBS_NAME, DBS_JOINING_PARAMS, "Dynamic Beam Shaping", elec, gas))

## LCIA results

Impacts, foreground electricity and consumable costs of each process in the minimum, amount and maximum scenarios.

In [ ]:
lcia_df = scenario_table([soa, dbs], impacts, prices)
export(lcia_df, units, RESULTS_DIR, f"{SOA_NAME}_{DBS_NAME}")
lcia_df

In [ ]:
all_units = {**units, ELECTRICITY_COLUMN: "kWh", COST_COLUMN: "EUR"}
indicator_cols = [c for c in lcia_df.columns if c not in ("Database", "Process", "Key", "Scenario")]
display(plots.scenario_bars(lcia_df, list(SOA_JOINING_PARAMS), [SOA_NAME, DBS_NAME], all_units,
                            technology_of, indicator_cols))

## Regional analysis: EU electricity markets

Replaces the RER electricity market with the market of each member state. The map shows the impacts and consumable costs per joined unit.
The file `results/regional_joining.csv` is an input of `workflow_assembly.ipynb`.

In [ ]:
markets = electricity_markets(ei, EU27)
regional_df = regional_table([soa, dbs], impacts, markets, ELEC_PRICE, prices, REGIONAL_BASELINE)
regional_df["technology"] = regional_df["process"].map(technology_of)
regional_df.to_csv(f"{RESULTS_DIR}/regional_joining.csv", index=False)
regional_df.head()

In [ ]:
amount_df = regional_df[regional_df["scenario"] == "amount"]
display(plots.regional_explorer(
    amount_df,
    filters=[("Use case", "key", list(SOA_JOINING_PARAMS), None),
             ("Technology", "database", [SOA_NAME, DBS_NAME], None)],
    units=all_units,
    title=lambda f: f"{f['key']}, {f['database'].replace('_', ' ')}: per joined unit",
    baseline=REGIONAL_BASELINE,
))